# First-Step-Contraction Replacement With RL Upstream Controller

This notebook mirrors the safety-filter RL workflow, but replaces QCQP projection with a first-step-contraction MPC solve. The TD3 policy still provides the candidate input, and the debug/export pipeline records when that candidate is accepted or replaced.

In [ ]:
import os
import numpy as np
import torch

from TD3Agent.agent import TD3Agent
from TD3Agent.reward_functions import make_reward_fn_mpc_quadratic
from Simulation.mpc import MpcSolver, compute_observer_gain
from Simulation.run_rl_lyapunov import run_rl_train
from Simulation.system_functions import PolymerCSTR
from Lyapunov.safety_debug import build_safety_filter_run_bundle, save_safety_filter_debug_artifacts
from utils.scaling_helpers import apply_min_max
from utils.td3_helpers import load_and_prepare_system_data

In [ ]:
# Polymer CSTR setup
Ad = 2.142e17
Ed = 14897
Ap = 3.816e10
Ep = 3557
At = 4.50e12
Et = 843
fi = 0.6
m_delta_H_r = -6.99e4
hA = 1.05e6
rhocp = 1506
rhoccpc = 4043
Mm = 104.14
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm])

CIf = 0.5888
CMf = 8.6981
Qi = 108.0
Qs = 459.0
Tf = 330.0
Tcf = 295.0
V = 3000.0
Vc = 3312.4
system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc])

Qm_ss = 378.0
Qc_ss = 471.6
system_steady_state_inputs = np.array([Qc_ss, Qm_ss])
delta_t = 0.5

steady_states = {"ss_inputs": system_steady_state_inputs.copy()}
cstr_ss = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t, deviation_form=False)
steady_states["y_ss"] = cstr_ss.y_ss.copy()
# Match the original rollout convention: the plant object stays in physical coordinates.
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)

u_min = np.array([71.6, 78.0])
u_max = np.array([870.0, 670.0])
setpoint_y_range_phys = np.array([[2.8, 320.0], [5.0, 326.0]])
setpoint_y_phys = np.array([[4.5, 324.0], [3.4, 321.0]])

system_dict_path = os.path.join("Data", "system_dict")
augmentation_style = "legacy"
augmentation_mode = None

system_data = load_and_prepare_system_data(
    steady_states=steady_states,
    setpoint_y=setpoint_y_range_phys,
    u_min=u_min,
    u_max=u_max,
    system_dict_path=system_dict_path,
    augmentation_style=augmentation_style,
    augmentation_mode=augmentation_mode,
)

A = system_data["A"]
B = system_data["B"]
C = system_data["C"]
A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
Bd_used = system_data["Bd_used"]
Cd_used = system_data["Cd_used"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]
min_max_dict["x_max"] = np.array([256.79686253, 256.01560603, 48.99447186, 144.79949103,
          2.82199733, 3.14014989, 2.78866348, 3.71691422,
          6.2029936])
min_max_dict["x_min"] = np.array([-272.28060121, -1112.33972595, -76.63993491, -608.60327886,
          -3.94399122, -3.93115257, -2.9532091, -4.06547624,
          -28.25906582])

inputs_number = int(B_aug.shape[1])
y_sp_scenario = apply_min_max(setpoint_y_phys, data_min[inputs_number:], data_max[inputs_number:]) - apply_min_max(
    steady_states["y_ss"],
    data_min[inputs_number:],
    data_max[inputs_number:],
)

n_tests = 200
set_points_len = 400
TEST_CYCLE = [False, False]
warm_start = 0
ACTOR_FREEZE = 0 * set_points_len
warm_start_plot = warm_start * 2 * set_points_len + ACTOR_FREEZE

poles = np.array([0.44619852, 0.33547649, 0.36380595, 0.70467118, 0.3562966,
                  0.42900673, 0.4228262, 0.96916776, 0.91230187])
L = compute_observer_gain(A_aug, C_aug, poles)

In [ ]:
# TD3 and MPC configuration
set_points_number = int(C_aug.shape[0])
inputs_number = int(B_aug.shape[1])
STATE_DIM = int(A_aug.shape[0]) + set_points_number + inputs_number
ACTION_DIM = int(B_aug.shape[1])

n_outputs = C_aug.shape[0]
ACTOR_LAYER_SIZES = [512, 512, 512, 512, 512]
CRITIC_LAYER_SIZES = [512, 512, 512, 512, 512]
BUFFER_CAPACITY = 40000
ACTOR_LR = 5e-5
CRITIC_LR = 5e-4
SMOOTHING_STD = 0.005
NOISE_CLIP = 0.01
GAMMA = 0.995
TAU = 0.005
MAX_ACTION = 1
POLICY_DELAY = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 256
STD_START = 0.02
STD_END = 0.001
STD_DECAY_RATE = 0.99992
STD_DECAY_MODE = "exp"

td3_agent = TD3Agent(
    state_dim=STATE_DIM,
    action_dim=ACTION_DIM,
    actor_hidden=ACTOR_LAYER_SIZES,
    critic_hidden=CRITIC_LAYER_SIZES,
    gamma=GAMMA,
    actor_lr=ACTOR_LR,
    critic_lr=CRITIC_LR,
    batch_size=BATCH_SIZE,
    policy_delay=POLICY_DELAY,
    target_policy_smoothing_noise_std=SMOOTHING_STD,
    noise_clip=NOISE_CLIP,
    max_action=MAX_ACTION,
    tau=TAU,
    std_start=STD_START,
    std_end=STD_END,
    std_decay_rate=STD_DECAY_RATE,
    std_decay_mode=STD_DECAY_MODE,
    buffer_size=BUFFER_CAPACITY,
    device=DEVICE,
    actor_freeze=ACTOR_FREEZE,
)

agent_path = os.path.join(os.getcwd(), "Data", "agent_2507171027.pkl")
if not os.path.exists(agent_path):
    raise FileNotFoundError(f"TD3 checkpoint not found: {agent_path}")
td3_agent.load(agent_path)

predict_h = 9
cont_h = 3
u_ss = apply_min_max(steady_states["ss_inputs"], data_min[:inputs_number], data_max[:inputs_number])
b_min = apply_min_max(u_min, data_min[:inputs_number], data_max[:inputs_number])
b_max = apply_min_max(u_max, data_min[:inputs_number], data_max[:inputs_number])
b1 = (b_min[0] - u_ss[0], b_max[0] - u_ss[0])
b2 = (b_min[1] - u_ss[1], b_max[1] - u_ss[1])
bnds = (b1, b2) * cont_h
cons = []
IC_opt = np.zeros(inputs_number * cont_h)

MPC_obj = MpcSolver(
    A_aug,
    B_aug,
    C_aug,
    Q_out=np.array([5.0, 1.0]),
    R_in=np.array([1.0, 1.0]),
    NP=predict_h,
    NC=cont_h,
)

reward_config, reward_fn = make_reward_fn_mpc_quadratic(
    Q_diag=np.array([5.0, 1.0]),
    R_diag=np.array([1.0, 1.0]),
)

nominal_qs = 459.0
nominal_qi = 108.0
nominal_hA = 1.05e6
qi_change = 0.95
qs_change = 1.05
ha_change = 0.92

Qs_tgt_diag = 1e8 * np.asarray(MPC_obj.Q_out, float)
Ru_tgt_diag = 1.0 * np.ones(MPC_obj.B.shape[1])

In [ ]:
# Run RL upstream + first-step-contraction replacement
selector_config = {
    "alpha_u_ref": 0.5,
    "alpha_du_sel": 0.5,
    "alpha_dx_sel": 0.05,
    "alpha_x_ref": 0.01,
    "x_weight_base": "CtQC",
    "use_output_bounds_in_selector": True,
}

run_config = {
    "rho_lyap": 0.98,
    "lyap_eps": 1e-9,
    "lyap_tol": 1e-10,
    "w_rl": 1.0,
    "w_track": 1.0,
    "w_move": 0.2,
    "w_ss": 0.1,
    "mpc_target_policy": "raw_setpoint",
    "tracking_target_policy": "raw_setpoint",
    "projection_backend": "first_step_contraction_mpc",
    "target_selector_config": selector_config,
    "selector_H": None,
    "target_backup_policy": "last_valid",
    "selector_warm_start": True,
    "reuse_mpc_solution_as_ic": False,
    "reset_system_on_entry": True,
    "first_step_contraction_on": True,
}
print("Using refined Step A selector with first-step-contraction MPC replacement backend")

# Recreate the plant before each run so repeated executions start from the nominal physical state.
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t, deviation_form=False)

results = run_rl_train(
    system=cstr,
    y_sp_scenario=y_sp_scenario,
    n_tests=n_tests,
    set_points_len=set_points_len,
    steady_states=steady_states,
    min_max_dict=min_max_dict,
    agent=td3_agent,
    MPC_obj=MPC_obj,
    L=L,
    data_min=data_min,
    data_max=data_max,
    warm_start=warm_start,
    test_cycle=TEST_CYCLE,
    nominal_qi=nominal_qi,
    nominal_qs=nominal_qs,
    nominal_ha=nominal_hA,
    qi_change=qi_change,
    qs_change=qs_change,
    ha_change=ha_change,
    reward_fn=reward_fn,
    mode="disturb",
    use_lyap=True,
    rho_lyap=run_config["rho_lyap"],
    lyap_eps=run_config["lyap_eps"],
    lyap_tol=run_config["lyap_tol"],
    w_rl=run_config["w_rl"],
    w_track=run_config["w_track"],
    w_move=run_config["w_move"],
    w_ss=run_config["w_ss"],
    Qy_track_diag=None,
    Rmove_diag=None,
    Qs_tgt_diag=Qs_tgt_diag,
    Ru_tgt_diag=Ru_tgt_diag,
    IC_opt=IC_opt,
    bnds=bnds,
    cons=cons,
    target_selector_config=run_config["target_selector_config"],
    selector_H=run_config["selector_H"],
    mpc_target_policy=run_config["mpc_target_policy"],
    tracking_target_policy=run_config["tracking_target_policy"],
    target_backup_policy=run_config["target_backup_policy"],
    selector_warm_start=run_config["selector_warm_start"],
    reuse_mpc_solution_as_ic=run_config["reuse_mpc_solution_as_ic"],
    reset_system_on_entry=run_config["reset_system_on_entry"],
    projection_backend=run_config["projection_backend"],
    first_step_contraction_on=run_config["first_step_contraction_on"],
    seed=0,
)

bundle = build_safety_filter_run_bundle(
    source="rl_first_step_replacement",
    results=results,
    steady_states=steady_states,
    config=run_config,
    min_max_dict=min_max_dict,
    data_min=data_min,
    data_max=data_max,
    extra={
        "reward_config": reward_config,
        "agent_path": agent_path,
        "delta_t": delta_t,
        "warm_start_plot": warm_start_plot,
        "start_plot_idx": 10,
        "actor_losses": list(td3_agent.actor_losses),
        "critic_losses": list(td3_agent.critic_losses),
    },
)

debug_dir = save_safety_filter_debug_artifacts(
    bundle=bundle,
    directory=os.path.join(os.getcwd(), "Data", "debug_exports"),
    prefix_name="rl_first_step_replacement",
    save_plots=True,
)

bundle["summary"], debug_dir